# DA6401 — System 1: Dropout 0.0, 0.2 + Localization + Feature Maps

**BEFORE RUNNING:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **ON**
3. Click **Save Version** → **Quick Save** to run in background (~8 hrs)

**What this notebook does:**
- §2.2 Dropout experiment: p=0.0, p=0.2
- Task 2: Localization (with pretrained encoder from classifier)
- §2.4 Feature map visualization
- §2.5 Detection table (auto-logged during localization training)

**W&B tag:** `kaggle-sys1`

**Output files:**
- `checkpoints/classifier.pth` — best classifier
- `checkpoints/classifier_drop00.pth` — dropout=0.0 best
- `checkpoints/classifier_drop02.pth` — dropout=0.2 best
- `checkpoints/localizer.pth` — best localizer

In [ ]:
import os
os.environ["WANDB_API_KEY"]      = "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"
os.environ["WANDB_DISABLE_CODE"] = "1"

CKPT = "/kaggle/working/checkpoints"
os.makedirs(CKPT, exist_ok=True)

!git clone https://github.com/usnaveen/A2_Deep_Learning.git /kaggle/working/repo 2>&1 | tail -2
%cd /kaggle/working/repo
!pip install -q wandb albumentations gdown scikit-learn

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
print(f"GPU: {gpu}")
assert torch.cuda.is_available(), "NO GPU — go to Settings and enable P100!"

In [ ]:
from data.pets_dataset import download_oxford_pet
download_oxford_pet("./data/oxford_pet")
print("Dataset ready.")

## 1. Classification — Dropout Experiments

In [ ]:
# ── §2.2  Dropout = 0.0 ──
!python train.py --task classify --experiment kaggle-sys1 --dropout 0.0 \
    --epochs 120 --lr 1e-3 --batch-size 32 --num-workers 2 \
    --checkpoint-dir {CKPT}

# Backup this run's best checkpoint
!cp {CKPT}/classifier.pth {CKPT}/classifier_drop00.pth
print("\n--- Backed up → classifier_drop00.pth ---")

In [ ]:
# ── §2.2  Dropout = 0.2 ──
!python train.py --task classify --experiment kaggle-sys1 --dropout 0.2 \
    --epochs 120 --lr 1e-3 --batch-size 32 --num-workers 2 \
    --checkpoint-dir {CKPT}

# Backup this run's best checkpoint
!cp {CKPT}/classifier.pth {CKPT}/classifier_drop02.pth
print("\n--- Backed up → classifier_drop02.pth ---")

## 2. Localization — Task 2
Uses the classifier's pretrained encoder from the best dropout run above.

In [ ]:
# ── §2.5  Localization (detection table auto-logged at end) ──
!python train.py --task localize --experiment kaggle-sys1 \
    --epochs 120 --lr 1e-3 --batch-size 32 --num-workers 2 \
    --checkpoint-dir {CKPT}

print("\n--- Localization done → localizer.pth ---")

## 3. Feature Map Visualization — §2.4

In [ ]:
# ── §2.4  Feature maps from first and last conv layers ──
!python inference.py --mode featuremaps --num-workers 2 --checkpoint-dir {CKPT}

print("\n--- Feature maps logged to W&B ---")

## 4. Summary

In [ ]:
import os, torch
from sklearn.metrics import f1_score
from data.pets_dataset import create_dataloaders
from models.classification import VGG11Classifier
from models.localization import VGG11Localizer
from train import compute_iou

CKPT = "/kaggle/working/checkpoints"
device = torch.device("cuda")
_, val_loader, _ = create_dataloaders(root="./data/oxford_pet", batch_size=64, num_workers=2)

print("="*60)
print("  SYSTEM 1 — FINAL SUMMARY")
print("="*60)

# Check all classifier variants
for name, drop in [("classifier_drop00.pth", 0.0), ("classifier_drop02.pth", 0.2)]:
    path = f"{CKPT}/{name}"
    if os.path.exists(path):
        model = VGG11Classifier(num_classes=37, dropout_p=drop, use_bn=True).to(device)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=False))
        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for b in val_loader:
                preds.extend(model(b["image"].to(device)).argmax(1).cpu().tolist())
                labels.extend(b["label"].tolist())
        f1 = f1_score(labels, preds, average="macro")
        print(f"  {name:30s}  Val F1 = {f1:.4f}")

# Check localizer
loc_path = f"{CKPT}/localizer.pth"
if os.path.exists(loc_path):
    model = VGG11Localizer().to(device)
    model.load_state_dict(torch.load(loc_path, map_location=device, weights_only=False))
    model.eval()
    total_iou, total = 0.0, 0
    with torch.no_grad():
        for b in val_loader:
            pred = model(b["image"].to(device)).cpu()
            ious = compute_iou(pred, b["bbox"])
            total_iou += ious.sum().item()
            total += len(ious)
    print(f"  {'localizer.pth':30s}  Val IoU = {total_iou/total:.4f}")

print("\n  Files in /kaggle/working/checkpoints/:")
for f in sorted(os.listdir(CKPT)):
    size_mb = os.path.getsize(f"{CKPT}/{f}") / 1e6
    print(f"    {f:40s} {size_mb:6.1f} MB")

print("\n  Download these from the Output tab after the session ends!")
print("="*60)